# Lab 03: Clarity Scoring Engine

## Business Context

The analysts' core hypothesis: **companies that can't explain themselves clearly in their S-1 filing tend to underperform.** To test this, we need a scoring engine that rates each section of an S-1 for messaging clarity on a 1-100 scale.

This lab builds that engine using LLM-as-judge, wraps the full agent in the **ChatAgent** interface required for Databricks Model Serving, and registers it in Unity Catalog.

| Attribute | Detail |
|---|---|
| **Key Skills** | LLM-as-judge, ChatAgent interface, intent routing, MLflow model registration |
| **Estimated Time** | 35 minutes |
| **Estimated Cost** | ~$2-3 |

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-core langchain-community langgraph unitycatalog-ai[databricks] unitycatalog-langchain databricks-agents --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG = "ipo_analyzer"
SCHEMA = "default"
LLM_ENDPOINT = "databricks-meta-llama-3.1-405b-instruct"

from databricks.vector_search.client import VectorSearchClient
from langchain_core.tools import tool
from langchain_community.chat_models import ChatDatabricks
from langgraph.prebuilt import create_react_agent

VS_ENDPOINT = "ipo_analyzer_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.filing_chunks_index"

SYSTEM_PROMPT = (
    "You are an IPO Filing Analyzer for a financial research firm. "
    "You have access to S-1 filings from tech IPOs and stock performance data.\n\n"
    "Tools:\n"
    "- search_filings: Search S-1 text for relevant passages. Always include the company name.\n"
    "- get_filing_metadata: Look up filing statistics\n"
    "- get_stock_performance: Look up stock price performance post-IPO\n"
    "- score_clarity: Score a filing section's messaging clarity (1-100)\n"
    "- query_scored_database: Query pre-computed clarity scores joined with stock returns\n\n"
    "Guidelines:\n"
    "- Always use tools to gather data before answering. Never guess.\n"
    "- When searching filings, include the company name AND topic keywords.\n"
    "- Cite the source filing. Structure analysis with clear sections and specific details.\n"
    "- IMPORTANT: You provide financial ANALYSIS, not investment ADVICE."
)

vsc = VectorSearchClient()
_vs_index = vsc.get_index(VS_ENDPOINT, VS_INDEX)

COMPANY_TICKERS = {
    'snowflake': 'SNOW', 'palantir': 'PLTR', 'doordash': 'DASH',
    'coinbase': 'COIN', 'rivian': 'RIVN', 'unity': 'U',
    'roblox': 'RBLX', 'bumble': 'BMBL', 'affirm': 'AFRM',
    'robinhood': 'HOOD', 'toast': 'TOST', 'confluent': 'CFLT',
    'gitlab': 'GTLB', 'hashicorp': 'HCP', 'duolingo': 'DUOL',
    'instacart': 'CART', 'klaviyo': 'KVYO', 'arm': 'ARM',
    'reddit': 'RDDT', 'rubrik': 'RBRK', 'astera': 'ALAB',
    'snow': 'SNOW', 'pltr': 'PLTR', 'dash': 'DASH', 'coin': 'COIN',
    'rivn': 'RIVN', 'rblx': 'RBLX', 'bmbl': 'BMBL', 'afrm': 'AFRM',
    'hood': 'HOOD', 'tost': 'TOST', 'cflt': 'CFLT', 'gtlb': 'GTLB',
    'hcp': 'HCP', 'duol': 'DUOL', 'cart': 'CART', 'kvyo': 'KVYO',
    'rddt': 'RDDT', 'rbrk': 'RBRK', 'alab': 'ALAB',
}

def _extract_tickers(query):
    found = []
    q_lower = query.lower()
    for name, ticker in COMPANY_TICKERS.items():
        if name in q_lower and ticker not in found:
            found.append(ticker)
    return found

def _extract_keywords(query):
    stopwords = {'what', 'did', 'say', 'about', 'in', 'their', 'the', 'and', 'how',
                 'does', 'do', 'compare', 'between', 'from', 'filing', 'filings',
                 's-1', 's1', 'ipo', 'for', 'with', 'that', 'this', 'are', 'was',
                 'were', 'has', 'have', 'had', 'been', 'their', 'they', 'its'}
    company_words = set(COMPANY_TICKERS.keys())
    words = query.lower().replace('?', '').replace(',', '').replace('.', '').split()
    return [w for w in words if w not in stopwords and w not in company_words and len(w) > 2][:5]

@tool
def search_filings(query: str) -> str:
    """Search S-1 filing text for passages relevant to the query.
    Use this for questions about what companies said in their IPO filings.
    Include the company name in your query for best results."""
    tickers = _extract_tickers(query)
    keywords = _extract_keywords(query)
    all_parts = []
    seen_texts = set()

    if tickers:
        for ticker in tickers[:2]:
            # Stage 1: SQL keyword search for precise matches
            if keywords:
                like_clauses = ' OR '.join(
                    [f"LOWER(chunk_text) LIKE '%{kw}%'" for kw in keywords]
                )
                sql = f"""
                    SELECT chunk_text, path FROM {CATALOG}.{SCHEMA}.filing_chunks
                    WHERE LOWER(path) LIKE '%{ticker}%'
                      AND ({like_clauses})
                    ORDER BY chunk_index LIMIT 10
                """
                try:
                    for row in spark.sql(sql).collect():
                        text_key = row.chunk_text[:100]
                        if text_key not in seen_texts:
                            seen_texts.add(text_key)
                            source = row.path.split('/')[-1].replace('-S1.html', '')
                            all_parts.append(f'[Source: {source}]\n{row.chunk_text}')
                except Exception:
                    pass

            # Stage 2: Vector search for semantic matches
            results = _vs_index.similarity_search(
                query_text=query, columns=['chunk_text', 'path'],
                num_results=10, filters={'path LIKE': f'%{ticker}%'},
                query_type='HYBRID',
            )
            for doc in results.get('result', {}).get('data_array', []):
                text_key = doc[0][:100]
                if text_key not in seen_texts:
                    seen_texts.add(text_key)
                    source = doc[1].split('/')[-1].replace('-S1.html', '')
                    all_parts.append(f'[Source: {source}]\n{doc[0]}')
    else:
        results = _vs_index.similarity_search(
            query_text=query, columns=['chunk_text', 'path'],
            num_results=15, query_type='HYBRID',
        )
        for doc in results.get('result', {}).get('data_array', []):
            source = doc[1].split('/')[-1].replace('-S1.html', '')
            all_parts.append(f'[Source: {source}]\n{doc[0]}')

    return '\n\n---\n\n'.join(all_parts[:20]) if all_parts else 'No relevant passages found.'


from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit

_uc_fn_names = [
    f"{CATALOG}.{SCHEMA}.get_filing_metadata",
    f"{CATALOG}.{SCHEMA}.get_stock_performance",
]
_uc_client = DatabricksFunctionClient()
_uc_toolkit = UCFunctionToolkit(function_names=_uc_fn_names, client=_uc_client)
tools = [search_filings] + _uc_toolkit.tools

llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=1024, temperature=0.1)
agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
print(f"Agent loaded with {len(tools)} tools: {[t.name for t in tools]}")


## A. Design the Clarity Rubric

An **LLM-as-judge** uses a second LLM to evaluate the output quality of a primary model — or, in our case, to score text quality directly. The rubric defines what each score level means, anchoring the judge's decisions.

The rubric below scores S-1 filing sections on a 1-100 clarity scale. Four section types are scored independently because "clarity" means different things in a business description vs. risk factors.

In [ ]:
CLARITY_RUBRIC = """You are scoring an S-1 filing section for messaging clarity.

## Rubric (1-100 scale)

- **1-20 (Impenetrable):** Dense jargon, circular definitions, no concrete specifics. A reader cannot explain what the company does after reading this section.
- **21-40 (Unclear):** Heavy jargon with occasional concrete details. A domain expert could piece it together, but a general investor would struggle.
- **41-60 (Adequate):** The core message is present but buried in boilerplate. Key details (numbers, timelines, specifics) are sparse.
- **61-80 (Clear):** A general investor can understand the main point. Concrete details are present. Some jargon remains but doesn't obscure meaning.
- **81-100 (Exceptional):** Plain language, specific numbers, clear cause-and-effect. A non-expert could explain this section to someone else accurately.

## Section type: {section_type}

## What to evaluate
- Business Description: What does the company do? For whom? How do they make money?
- Risk Factors: Are the risks specific to this company, or generic boilerplate?
- Competitive Landscape: Does the filing name competitors and explain differentiation?
- Revenue Model: Is the revenue breakdown clear (segments, growth drivers, unit economics)?

## Filing text to score:
{text}

## Instructions
Respond with ONLY a JSON object on a single line:
{{"score": <integer 1-100>, "justification": "<one sentence explaining the score>"}}"""

# Test the rubric manually with a sample passage
from langchain_community.chat_models import ChatDatabricks

judge_llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=200, temperature=0.0)

# Get a sample chunk from Snowflake's filing
sample = spark.sql(f"""
SELECT chunk_text FROM {CATALOG}.{SCHEMA}.filing_chunks
WHERE LOWER(path) LIKE '%snow%'
  AND LENGTH(chunk_text) > 1000
  AND (LOWER(chunk_text) LIKE '%business%' OR LOWER(chunk_text) LIKE '%revenue%' OR LOWER(chunk_text) LIKE '%platform%')
LIMIT 1
""").first()["chunk_text"]

prompt = CLARITY_RUBRIC.format(section_type="business_description", text=sample[:3000])
result = judge_llm.invoke(prompt)
print("Sample scoring result:")
print(result.content)

## B. Create score_clarity UC Function

Wrapping the scoring logic in a UC function makes it:
- **Callable by the agent** via UCFunctionToolkit
- **Callable from SQL** via `ai_query()` for batch scoring (Lab 06)
- **Governed** by UC permissions
- **Versioned** in the catalog

In [ ]:
spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.score_clarity")

spark.sql(f"""
CREATE FUNCTION {CATALOG}.{SCHEMA}.score_clarity(
    filing_text STRING,
    section_type STRING
)
RETURNS STRING
COMMENT 'Score an S-1 filing section for messaging clarity (1-100). Returns JSON with score and justification. Valid section_type values: business_description, risk_factors, competitive_landscape, revenue_model.'
RETURN ai_query(
    '{LLM_ENDPOINT}',
    CONCAT(
        'You are scoring an S-1 filing section for messaging clarity on a 1-100 scale.\\n\\n',
        'Rubric:\\n',
        '1-20 (Impenetrable): Dense jargon, circular definitions, no specifics.\\n',
        '21-40 (Unclear): Heavy jargon with occasional concrete details.\\n',
        '41-60 (Adequate): Core message present but buried in boilerplate.\\n',
        '61-80 (Clear): General investor can understand. Concrete details present.\\n',
        '81-100 (Exceptional): Plain language, specific numbers, clear cause-and-effect.\\n\\n',
        'Section type: ', section_type, '\\n\\n',
        'Filing text:\\n', SUBSTRING(filing_text, 1, 6000), '\\n\\n',
        'Respond with ONLY a JSON object: {{\"score\": <int>, \"justification\": \"<one sentence>\"}}'
    )
)::STRING
""")

print(f"Created: {CATALOG}.{SCHEMA}.score_clarity")

# Test it
display(spark.sql(f"""
SELECT {CATALOG}.{SCHEMA}.score_clarity(
    (SELECT chunk_text FROM {CATALOG}.{SCHEMA}.filing_chunks WHERE LOWER(path) LIKE '%snow%' LIMIT 1),
    'business_description'
) AS clarity_score
"""))

## C. Intent Routing

Instead of letting the agent guess which tool to use, we classify the user's intent upfront and route to the appropriate tool subset. This reduces unnecessary tool calls and improves response quality.

| Intent | Description | Primary Tools |
|---|---|---|
| `RESEARCH` | Content or concept questions about filings | search_filings |
| `STOCK_LOOKUP` | Stock performance questions | get_stock_performance |
| `CLARITY_SCORE` | Score a section for clarity | score_clarity |
| `COMPARISON` | Cross-data analysis | search_filings + get_stock_performance |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

classifier_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an intent classifier for an IPO filing analyzer.\n"
     "Classify the user's query into exactly one of: RESEARCH, STOCK_LOOKUP, CLARITY_SCORE, COMPARISON.\n\n"
     "RESEARCH — questions about filing content (risk factors, business model, competition)\n"
     "STOCK_LOOKUP — questions about stock price or returns\n"
     "CLARITY_SCORE — requests to score or evaluate filing clarity\n"
     "COMPARISON — questions combining filing analysis with stock performance\n\n"
     "Respond with only the label."),
    ("human", "{query}"),
])

classifier_chain = classifier_prompt | llm | StrOutputParser()

# Test
test_queries = [
    ("RESEARCH",      "What are Snowflake's key risk factors?"),
    ("STOCK_LOOKUP",  "How did COIN perform in its first year?"),
    ("CLARITY_SCORE", "Score Coinbase's risk factors for clarity"),
    ("COMPARISON",    "Do companies with clearer business descriptions perform better?"),
]

for expected, query in test_queries:
    intent = classifier_chain.invoke({"query": query}).strip().upper()
    match = "OK" if intent == expected else "MISMATCH"
    print(f"  [{match}] '{query}' -> {intent} (expected {expected})")

## D. ChatAgent + UC Registration

Databricks Model Serving requires models to implement the **ChatAgent** interface. This ensures:
- Consistent request/response schema (OpenAI-compatible message format)
- Automatic tracing integration with MLflow
- Compatibility with the AI Gateway and Review App

`ChatAgent.predict(messages, context)` is the single required method.

In [ ]:
import uuid
try:
    from databricks.agents import ChatAgent, ChatAgentMessage, ChatAgentResponse
except ImportError:
    from mlflow.pyfunc import ChatAgent
    from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse
from typing import Optional

# Reload tools with scoring enabled
# Rebuild tools with scoring UC functions included
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit

_scoring_fn_names = [
    f"{CATALOG}.{SCHEMA}.get_filing_metadata",
    f"{CATALOG}.{SCHEMA}.get_stock_performance",
    f"{CATALOG}.{SCHEMA}.score_clarity",
]
# query_scored_database is created in Lab 06 — add if it exists
try:
    spark.sql(f"DESCRIBE FUNCTION {CATALOG}.{SCHEMA}.query_scored_database")
    _scoring_fn_names.append(f"{CATALOG}.{SCHEMA}.query_scored_database")
except Exception:
    pass

_scoring_client = DatabricksFunctionClient()
_scoring_toolkit = UCFunctionToolkit(function_names=_scoring_fn_names, client=_scoring_client)
all_tools_with_scoring = [search_filings] + _scoring_toolkit.tools

INTENT_PROMPTS = {
    "RESEARCH": (
        "You are an IPO Filing Analyzer. Use search_filings to find relevant passages, "
        "then synthesize a grounded answer with citations."
    ),
    "STOCK_LOOKUP": (
        "You are a stock data assistant. Use get_stock_performance to look up the requested ticker. "
        "Report the numbers clearly. Do NOT give investment advice."
    ),
    "CLARITY_SCORE": (
        "You are a filing clarity scorer. Use search_filings to find the relevant section, "
        "then use score_clarity to score it. Report the score and justification."
    ),
    "COMPARISON": (
        "You are an IPO analyst. Use search_filings and get_stock_performance to gather data, "
        "then compare and analyze. Ground every claim in the data. Do NOT give investment advice."
    ),
}


class IpoAnalyzerAgent(ChatAgent):
    """ChatAgent wrapping the IPO Filing Analyzer with intent routing."""

    def __init__(self):
        from langchain_community.chat_models import ChatDatabricks
        from langgraph.prebuilt import create_react_agent
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.output_parsers import StrOutputParser

        self._llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=1024, temperature=0.1)
        self._tools = all_tools_with_scoring
        self._classifier = (
            ChatPromptTemplate.from_messages([
                ("system",
                 "Classify into: RESEARCH, STOCK_LOOKUP, CLARITY_SCORE, COMPARISON.\n"
                 "Respond with only the label."),
                ("human", "{query}"),
            ]) | self._llm | StrOutputParser()
        )

    def predict(self, messages: list[ChatAgentMessage], context: Optional[dict] = None, custom_inputs: Optional[dict] = None) -> ChatAgentResponse:
        from langgraph.prebuilt import create_react_agent

        user_query = messages[-1].content if messages else ""

        intent = self._classifier.invoke({"query": user_query}).strip().upper()
        if intent not in INTENT_PROMPTS:
            intent = "RESEARCH"

        routed_agent = create_react_agent(
            self._llm, self._tools, prompt=INTENT_PROMPTS[intent]
        )
        result = routed_agent.invoke(
            {"messages": [{"role": "user", "content": user_query}]}
        )

        return ChatAgentResponse(
            messages=[
                ChatAgentMessage(
                    role="assistant",
                    id=str(uuid.uuid4()),
                    content=result["messages"][-1].content,
                )
            ]
        )


# Smoke test
try:
    agent_obj = IpoAnalyzerAgent()
    test_msg = ChatAgentMessage(role="user", id=str(uuid.uuid4()), content="Score Snowflake's business description for clarity.")
    response = agent_obj.predict([test_msg])
    print("ChatAgent response:")
    print(response.messages[0].content)
    
except Exception as e:
    print(f'ChatAgent smoke test skipped: {e}')

In [ ]:
import mlflow
import shutil

username = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{username}/ipo-analyzer/lab-03-clarity-engine")

# Copy agent model file to /tmp for code-based logging
# The agent_model.py file defines IpoAnalyzerAgent (ChatAgent subclass)
agent_file = "/tmp/ipo_agent_model.py"
try:
    # Try workspace path first (if running from workspace)
    import os
    nb_dir = os.path.dirname(os.path.abspath("__file__"))
    candidates = [
        "/Workspace/Users/btriani@gmail.com/ipo-analyzer/agent_model.py",
        os.path.join(nb_dir, "agent_model.py"),
    ]
    for c in candidates:
        if os.path.exists(c):
            shutil.copy(c, agent_file)
            print(f"Copied from {c}")
            break
    else:
        raise FileNotFoundError("agent_model.py not found in expected locations")
except Exception as e:
    print(f"Note: {e}")
    print("Writing agent_model.py inline...")
    # Inline fallback: write minimal agent file
    with open(agent_file, "w") as f:
        f.write("import uuid\n")
        f.write("from typing import Optional\n")
        f.write("try:\n")
        f.write("    from databricks.agents import ChatAgent, ChatAgentMessage, ChatAgentResponse\n")
        f.write("except ImportError:\n")
        f.write("    from mlflow.pyfunc import ChatAgent\n")
        f.write("    from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse\n")
        f.write("import mlflow\n")
        f.write("from langchain_community.chat_models import ChatDatabricks\n")
        f.write("from langgraph.prebuilt import create_react_agent\n")
        f.write("from langchain_core.tools import tool\n")
        f.write("from databricks.vector_search.client import VectorSearchClient\n")
        f.write("from unitycatalog.ai.core.databricks import DatabricksFunctionClient\n")
        f.write("from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit\n")
        f.write("from langchain_core.prompts import ChatPromptTemplate\n")
        f.write("from langchain_core.output_parsers import StrOutputParser\n")
        f.write("\n")
        f.write("class IpoAnalyzerAgent(ChatAgent):\n")
        f.write("    def __init__(self):\n")
        f.write("        self._llm = ChatDatabricks(endpoint='databricks-meta-llama-3.1-405b-instruct', max_tokens=1024, temperature=0.1)\n")
        f.write("        vsc = VectorSearchClient()\n")
        f.write("        self._vs = vsc.get_index('ipo_analyzer_vs_endpoint', 'ipo_analyzer.default.filing_chunks_index')\n")
        f.write("        @tool\n")
        f.write("        def search_filings(query: str) -> str:\n")
        f.write("            'Search S-1 filings for relevant passages.'\n")
        f.write("            r = self._vs.similarity_search(query_text=query, columns=['chunk_text','path'], num_results=10, query_type='HYBRID')\n")
        f.write("            docs = r.get('result',{}).get('data_array',[])\n")
        f.write("            return '\\n---\\n'.join(['['+d[1].split('/')[-1]+'] '+d[0] for d in docs]) or 'None found.'\n")
        f.write("        uc = UCFunctionToolkit(function_names=['ipo_analyzer.default.get_filing_metadata','ipo_analyzer.default.get_stock_performance','ipo_analyzer.default.score_clarity'], client=DatabricksFunctionClient())\n")
        f.write("        self._tools = [search_filings] + uc.tools\n")
        f.write("    def predict(self, messages, context=None, custom_inputs=None):\n")
        f.write("        q = messages[-1].content if messages else ''\n")
        f.write("        agent = create_react_agent(self._llm, self._tools, prompt='You are an IPO Filing Analyzer.')\n")
        f.write("        r = agent.invoke({'messages': [{'role':'user','content':q}]})\n")
        f.write("        return ChatAgentResponse(messages=[ChatAgentMessage(role='assistant',id=str(uuid.uuid4()),content=r['messages'][-1].content)])\n")
        f.write("\n")
        f.write("mlflow.models.set_model(IpoAnalyzerAgent())\n")

# Verify syntax
import py_compile
py_compile.compile(agent_file, doraise=True)
print(f"Agent file ready: {agent_file} (syntax OK)")

with mlflow.start_run(run_name="ipo-analyzer-v1") as run:
    mlflow.log_params({
        "llm_endpoint": LLM_ENDPOINT,
        "rubric_version": "v1",
        "intent_classes": "RESEARCH,STOCK_LOOKUP,CLARITY_SCORE,COMPARISON",
    })

    model_info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model=agent_file,
        pip_requirements=[
            "databricks-sdk", "databricks-vectorsearch", "mlflow",
            "langchain", "langchain-community", "langgraph",
            "unitycatalog-langchain", "databricks-agents",
        ],
    )
    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")

mlflow.set_registry_uri("databricks-uc")
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.ipo_filing_agent"
registered = mlflow.register_model(model_uri=model_info.model_uri, name=UC_MODEL_NAME)
print(f"Registered: {registered.name} v{registered.version}")

## Before / After

**Before:** The agent can answer questions and look up stock data, but it can't evaluate *how clearly* a company communicates. Asking "Score Coinbase's risk factors for clarity" gets a confused response.

**After:** The agent scores any S-1 section on a 1-100 clarity scale with a justification. It's wrapped in ChatAgent and registered in Unity Catalog — ready for deployment.

In [ ]:
# The clarity scoring question
print("=== CLARITY SCORING ===")
test = agent_obj.predict([
    ChatAgentMessage(role="user", id=str(uuid.uuid4()),
                     content="Score Coinbase's risk factors section for clarity.")
])
print(test.messages[0].content)

print()

# Intent routing in action
for q in [
    "What does Rivian say about its market opportunity?",
    "How did RBLX perform in the first year?",
    "Score DoorDash's revenue model for clarity",
]:
    intent = classifier_chain.invoke({"query": q}).strip().upper()
    print(f"  [{intent}] {q}")

In [ ]:
# Return key results via notebook exit
import json as _json
_results = {}
try:
    # Clarity rubric test result
    _results["clarity_score_raw"] = result.content[:400] if hasattr(result, "content") else str(result)[:400]
    # Intent classifier results
    _results["intent_tests"] = "check notebook for details"
    # Model registration
    _results["model_registered"] = str(registered.name) + " v" + str(registered.version)
    # ChatAgent smoke test
    try:
        _results["chatagent_test"] = response.messages[0].content[:400]
    except Exception:
        _results["chatagent_test"] = "skipped (non-blocking)"
except Exception as e:
    _results["error"] = str(e)[:300]
dbutils.notebook.exit(_json.dumps(_results))


In [ ]:
# Scorecard -- tracks cumulative progress
try:
    import sys
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _parent = "/Workspace" + "/".join(_nb_path.split("/")[:-1])
    if _parent not in sys.path:
        sys.path.insert(0, _parent)
    from shared.lab_utils import get_scorecard
    get_scorecard()
except Exception as e:
    print(f"Scorecard skipped: {e}")